In [ ]:
!pip install pandas scikit-learn

In [ ]:
import pandas as pd

# Load the dataset
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Path to your dataset in Google Drive
file_path = '/content/drive/MyDrive/Dataset/dataset.csv'

# Load the dataset
data = pd.read_csv(file_path)

# Display the first few rows
data.head()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [ ]:

# Fill missing numerical values with the median
data['LoanAmount'] = data['LoanAmount'].fillna(data['LoanAmount'].median())

# Fill missing values in Credit_History using the mode
data['Credit_History'] = data['Credit_History'].fillna(data['Credit_History'].mode().iloc[0])

# Fill missing values in Loan_Amount_Term using the mode
data['Loan_Amount_Term'] = data['Loan_Amount_Term'].fillna(data['Loan_Amount_Term'].mode().iloc[0])


# Fill missing categorical values with the mode
data['Gender'] = data['Gender'].fillna(data['Gender'].mode().iloc[0])
data['Married'] = data['Married'].fillna(data['Married'].mode().iloc[0])
data['Dependents'] = data['Dependents'].fillna(data['Dependents'].mode().iloc[0])
data['Self_Employed'] = data['Self_Employed'].fillna(data['Self_Employed'].mode().iloc[0])
data['Education'] = data['Education'].fillna(data['Education'].mode().iloc[0])
data['Property_Area'] = data['Property_Area'].fillna(data['Property_Area'].mode().iloc[0])


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create label encoder
label_encoder = LabelEncoder()

# Encode categorical features
data['Gender'] = label_encoder.fit_transform(data['Gender'])
data['Married'] = label_encoder.fit_transform(data['Married'])
data['Self_Employed'] = label_encoder.fit_transform(data['Self_Employed'])
data['Education'] = label_encoder.fit_transform(data['Education'])
data['Property_Area'] = label_encoder.fit_transform(data['Property_Area'])
data['Loan_Status'] = label_encoder.fit_transform(data['Loan_Status'])  # Target variable


In [ ]:
from sklearn.preprocessing import StandardScaler

# Select numeric columns
numeric_features = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']

# Initialize the scaler
scaler = StandardScaler()

# Scale the numerical features
data[numeric_features] = scaler.fit_transform(data[numeric_features])


In [ ]:
# Split data into features (X) and target (y)
X = data.drop(columns=['Loan_ID', 'Loan_Status'])  # Drop Loan_ID as it's not a feature
y = data['Loan_Status']  # Target variable


In [ ]:
from sklearn.model_selection import train_test_split

# Split the data (80% for training and 20% for testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# Check the unique values in the 'Dependents' column
data['Dependents'].unique()


array(['0', '1', '2', '3+'], dtype=object)

In [ ]:
# Replace '3+' in 'Dependents' with 3 and convert to float
data['Dependents'] = data['Dependents'].replace('3+', 3).astype(float)


In [ ]:
# Convert all columns that should be numeric to float
numeric_columns = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Dependents']
for col in numeric_columns:
    data[col] = pd.to_numeric(data[col], errors='coerce')  # This will turn any problematic values to NaN


In [ ]:
# Fill NaN values with the median for numerical columns
data[numeric_columns] = data[numeric_columns].fillna(data[numeric_columns].median())


In [ ]:
# Split the data into features and target
X = data.drop(columns=['Loan_ID', 'Loan_Status'])  # Drop non-feature columns
y = data['Loan_Status']  # Target variable



In [ ]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the Logistic Regression model
model = LogisticRegression(max_iter=1000)

# Train the model
model.fit(X_train, y_train)

# Predict on the test data
y_pred = model.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.7886178861788617
Confusion Matrix:
 [[18 25]
 [ 1 79]]
Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.42      0.58        43
           1       0.76      0.99      0.86        80

    accuracy                           0.79       123
   macro avg       0.85      0.70      0.72       123
weighted avg       0.83      0.79      0.76       123



In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and train the Random Forest model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Predict and evaluate
y_pred_rf = rf_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))


Accuracy: 0.7560975609756098
Confusion Matrix:
 [[18 25]
 [ 5 75]]
Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.42      0.55        43
           1       0.75      0.94      0.83        80

    accuracy                           0.76       123
   macro avg       0.77      0.68      0.69       123
weighted avg       0.76      0.76      0.73       123



In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Calculate the imbalance ratio (for class 1 and class 0)
class_0 = (y == 0).sum()  # Number of "Not Approved" cases
class_1 = (y == 1).sum()  # Number of "Approved" cases

# Calculate the balance ratio (ratio of the number of "Not Approved" to "Approved")
balance_ratio = class_1 / class_0


# Initialize and train the XGBoost model
xgb_model = xgb.XGBClassifier(scale_pos_weight=balance_ratio, random_state=42)
xgb_model.fit(X_train, y_train)

# Predict and evaluate
y_pred_xgb = xgb_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))


Accuracy: 0.7560975609756098
Confusion Matrix:
 [[19 24]
 [ 6 74]]
Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.44      0.56        43
           1       0.76      0.93      0.83        80

    accuracy                           0.76       123
   macro avg       0.76      0.68      0.70       123
weighted avg       0.76      0.76      0.74       123



In [ ]:
from sklearn.svm import SVC

# Initialize and train the Support Vector Machine model
svm_model = SVC(class_weight='balanced', random_state=42)
svm_model.fit(X_train, y_train)

# Predict and evaluate
y_pred_svm = svm_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))
print("Classification Report:\n", classification_report(y_test, y_pred_svm))


Accuracy: 0.7560975609756098
Confusion Matrix:
 [[18 25]
 [ 5 75]]
Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.42      0.55        43
           1       0.75      0.94      0.83        80

    accuracy                           0.76       123
   macro avg       0.77      0.68      0.69       123
weighted avg       0.76      0.76      0.73       123



In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize and train the KNN model
knn_model = KNeighborsClassifier()
knn_model.fit(X_train, y_train)

# Predict and evaluate
y_pred_knn = knn_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_knn))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_knn))
print("Classification Report:\n", classification_report(y_test, y_pred_knn))


Accuracy: 0.7073170731707317
Confusion Matrix:
 [[12 31]
 [ 5 75]]
Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.28      0.40        43
           1       0.71      0.94      0.81        80

    accuracy                           0.71       123
   macro avg       0.71      0.61      0.60       123
weighted avg       0.71      0.71      0.66       123



In [ ]:
from sklearn.naive_bayes import GaussianNB

# Initialize and train the Naive Bayes model
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# Predict and evaluate
y_pred_nb = nb_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))
print("Classification Report:\n", classification_report(y_test, y_pred_nb))


Accuracy: 0.7804878048780488
Confusion Matrix:
 [[18 25]
 [ 2 78]]
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.42      0.57        43
           1       0.76      0.97      0.85        80

    accuracy                           0.78       123
   macro avg       0.83      0.70      0.71       123
weighted avg       0.81      0.78      0.75       123



In [ ]:
from keras.models import Sequential
from keras.layers import Dense

# Initialize and build the neural network model
ann_model = Sequential()
ann_model.add(Dense(units=64, activation='relu', input_dim=X_train.shape[1]))  # First hidden layer
ann_model.add(Dense(units=32, activation='relu'))  # Second hidden layer
ann_model.add(Dense(units=1, activation='sigmoid'))  # Output layer (binary classification)

# Compile the model
ann_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
ann_model.fit(X_train, y_train, epochs=10, batch_size=32)

# Predict and evaluate
y_pred_ann = (ann_model.predict(X_test) > 0.5).astype(int)  # Threshold prediction to binary
print("Accuracy:", accuracy_score(y_test, y_pred_ann))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_ann))
print("Classification Report:\n", classification_report(y_test, y_pred_ann))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6357 - loss: 0.6488
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7016 - loss: 0.5722 
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6677 - loss: 0.5786 
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7106 - loss: 0.5433 
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7654 - loss: 0.5096 
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7563 - loss: 0.5246  
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8052 - loss: 0.4825 
Epoch 8/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8200 - loss: 0.4675 
Epoch 9/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8358 - loss: 0.4585 
Epoch 10/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8469 - loss: 0.4305 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
Accuracy: 0.7886178861788617
Confusion Matrix:
 [[18 25]
 [ 1 79]]
Classification Report:
               precis

In [ ]:
!pip install lightgbm
import lightgbm as lgb
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Calculate the balance ratio (for class 1 and class 0)
class_0 = (y == 0).sum()  # Number of "Not Approved" cases
class_1 = (y == 1).sum()  # Number of "Approved" cases

# Calculate the balance ratio (ratio of the number of "Approved" to "Not Approved")
balance_ratio = class_1 / class_0

# Initialize and train the LightGBM model with scale_pos_weight
lgb_model = lgb.LGBMClassifier(scale_pos_weight=balance_ratio, random_state=42)

# Train the model
lgb_model.fit(X_train, y_train)

# Predict on the test data
y_pred_lgb = lgb_model.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred_lgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lgb))
print("Classification Report:\n", classification_report(y_test, y_pred_lgb))


[LightGBM] [Info] Number of positive: 342, number of negative: 149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000974 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 370
[LightGBM] [Info] Number of data points in the train set: 491, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.696538 -> initscore=0.830864
[LightGBM] [Info] Start training from score 0.830864
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[

In [ ]:
!pip install catboost
import catboost
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Calculate the balance ratio (for class 1 and class 0)
class_0 = (y == 0).sum()  # Number of "Not Approved" cases
class_1 = (y == 1).sum()  # Number of "Approved" cases

# Calculate the balance ratio (ratio of the number of "Approved" to "Not Approved")
balance_ratio = class_1 / class_0

# Initialize and train the CatBoost model with scale_pos_weight
catboost_model = CatBoostClassifier(
    scale_pos_weight=balance_ratio,
    iterations=1000,          # Number of boosting iterations
    depth=6,                  # Depth of the trees
    learning_rate=0.1,       # Learning rate
    cat_features=[],         # List of categorical features (if applicable)
    random_state=42,
    verbose=200              # Print logs every 200 iterations
)

# Train the model
catboost_model.fit(X_train, y_train)

# Predict on the test data
y_pred_catboost = catboost_model.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred_catboost))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_catboost))
print("Classification Report:\n", classification_report(y_test, y_pred_catboost))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 5.0 MB/s eta 0:00:00
0:	learn: 0.6119912	total: 47.9ms	remaining: 47.8s
200:	learn: 0.0999186	total: 234ms	remaining: 931ms
400:	learn: 0.0495289	total: 421ms	remaining: 628ms
600:	learn: 0.0314892	total: 603ms	remaining: 400ms
800:	learn: 0.0214847	total: 814ms	remaining: 202ms
999:	learn: 0.0167095	total: 999ms	remaining: 0us
Accuracy: 0.7804878048780488
Confusion Matrix:
 [[19 24]
 [ 3 77]]
Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.44      0.58        43
           1       0.76      0.96      0.85        80

    accuracy                           0.78       123
   macro avg       0.81      0.70      0.72       123
weighted avg       0.80      0.78      0.76       123

